<a href="https://colab.research.google.com/github/tamarakalashnyk20-glitch/ab_test_python/blob/main/ab_test_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[A/B Test Dashboard](https://public.tableau.com/views/ABTestfinal/Dashboard12?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

[A/b Test Results](https://drive.google.com/file/d/196sW4v_ySuPQ5mavTMVeLigwPDFbbFWj/view?usp=sharing)

[A/B Test Results by differend dimensions](https://drive.google.com/file/d/1m2X6ddCE3cdhWJgvseOVQF7h1Hhc2KGy/view?usp=sharing)

## Calculation of statistical significance

In [ ]:
#Libraries and BigQuery setup
!pip install statsmodels
from google.cloud import bigquery
from google.colab import auth
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from google.colab import files

In [ ]:
# Authentication
auth.authenticate_user()

# Creating a BigQuery client
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
query = """
with session_info as (
SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.continent,
      sp.device,
      sp.channel,
      ab.test,
      ab.test_group
from `DA.ab_test` as ab
join `DA.session` as s
on ab.ga_session_id = s.ga_session_id
join `DA.session_params` as sp
on ab.ga_session_id = sp.ga_session_id
),
session_with_orders as (
 select
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct o.ga_session_id) as session_with_orders
from `DA.order` as o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group
),
events as (
select
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      count(ep.ga_session_id) as event_cnt
from `DA.event_params` ep
join session_info
on ep.ga_session_id = session_info.ga_session_id
group by
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
),
session as (
select
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct session_info.ga_session_id) as session_cnt
from session_info
group by
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group
),
account as (
select
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct acs.ga_session_id) as new_account_cnt
from `DA.account_session` acs
join session_info
on acs.ga_session_id = session_info.ga_session_id
group by
      session_info.date,
      session_info.country,
      session_info.continent,
      session_info.device,
      session_info.channel,
      session_info.test,
      session_info.test_group
)


SELECT
      session_with_orders.date,
      session_with_orders.country,
      session_with_orders.continent,
      session_with_orders.device,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session with orders' as event_name,
      session_with_orders.session_with_orders as value
from session_with_orders
union all
SELECT
      events.date,
      events.country,
      events.continent,
      events.device,
      events.channel,
      events.test,
      events.test_group,
      events.event_name,
      events.event_cnt as value
from events
union all
SELECT
      session.date,
      session.country,
      session.continent,
      session.device,
      session.channel,
      session.test,
      session.test_group,
      'session' as event_name,
      session.session_cnt as value
from session
union all
SELECT
      account.date,
      account.country,
      account.continent,
      account.device,
      account.channel,
      account.test,
      account.test_group,
      'new account' as event_name,
      account.new_account_cnt as value
from account
"""
df = client.query(query).to_dataframe()
df.head()

In [ ]:
#Event Frequency Count
event_counts = df['event_name'].value_counts()
print(event_counts)

In [ ]:
unique_rows = df[["date", "country", "device", "continent", "channel", "test", "test_group", "event_name"]].drop_duplicates()

print("Total number of rows:", len(df))
print("Nubmer of unique rows:", len(unique_rows))

In [ ]:
#Metrics
metrics = [{"name": "begin_checkout/session", "num": "begin_checkout", "den": "session"},
           {"name": "add_payment_info/session", "num": "add_payment_info", "den": "session"},
           {"name": "add_shipping_info/session", "num": "add_shipping_info", "den": "session"},
           {"name": "new account/session", "num": "new account", "den": "session"}]
results = []

#Creating pivot DataFrame
def make_pivot(df):
        agg = df.groupby(["test", "test_group", "event_name"])["value"].sum().reset_index()

        pivot = agg.pivot_table(index=["test", "test_group"],
        columns="event_name", values="value", fill_value=0).reset_index()

        pivot.columns.name = None
        return pivot
pivot_total = make_pivot(df)

In [ ]:
# List to store results for all tests
results = []

# Get all unique test IDs
tests = pivot_total['test'].unique()

# Loop through each test
for test_id in tests:
    # Filter data for the current test
    df_test = pivot_total[pivot_total['test'] == test_id]

    # Split into two groups
    group1 = df_test[df_test['test_group'] == 1]
    group2 = df_test[df_test['test_group'] == 2]

    # Loop through each metric
    for metric in metrics:
        # Sum numerator and denominator for each group
        num1 = group1[metric['num']].sum()
        num2 = group2[metric['num']].sum()
        den1 = group1[metric['den']].sum()
        den2 = group2[metric['den']].sum()

        # Calculate conversion rates
        conv1 = num1 / den1 if den1 > 0 else 0
        conv2 = num2 / den2 if den2 > 0 else 0

        # Perform Z-test for proportions
        count = np.array([num1, num2])
        nobs = np.array([den1, den2])
        z_stat, p_value = proportions_ztest(count, nobs)

        # Calculate metric change and significance
        metric_change = conv2 / conv1 if conv1 > 0 else np.nan
        significant = p_value < 0.05

        # Append the results to the list
        results.append({
            "test_number": test_id,
            "metric": metric['name'],
            "numerator_ev": num1,
            "denominator_ev": den1,
            "numerator_cohort": num2,
            "denominator_cohort": den2,
            "conversion_rate_1": conv1,
            "conversion_rate_2": conv2,
            "metric_change": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": significant
        })

# Create a DataFrame from the collected results
df_results = pd.DataFrame(results)

print(df_results.head())

# Save results to CSV
df_results.to_csv("test_results.csv", index=False)

In [ ]:
# Main insights
df_results.sort_values(by="p_value").head(20)

## Calculation of statistical significance across breakdowns by countries, devices, continents, and traffic channels

In [ ]:
results = []

dimensions = ["continent", "device", "channel", "country"]

tests = df['test'].unique()

for dim in dimensions:
    for test_id in tests:

        dim_values = df[df['test'] == test_id][dim].dropna().unique()

        for dim_value in dim_values:

            df_slice = df[
                (df['test'] == test_id) &
                (df[dim] == dim_value)
            ]

            if df_slice.empty:
                continue

            pivot = make_pivot(df_slice)

            group1 = pivot[pivot['test_group'] == 1]
            group2 = pivot[pivot['test_group'] == 2]

            if group1.empty or group2.empty:
                continue

            for metric in metrics:
                num1 = group1[metric['num']].sum() if metric['num'] in group1.columns else 0
                num2 = group2[metric['num']].sum() if metric['num'] in group2.columns else 0
                den1 = group1[metric['den']].sum() if metric['den'] in group1.columns else 0
                den2 = group2[metric['den']].sum() if metric['den'] in group2.columns else 0


                if den1 < 30 or den2 < 30:
                    continue

                conv1 = num1 / den1
                conv2 = num2 / den2

                count = np.array([num1, num2])
                nobs = np.array([den1, den2])

                z_stat, p_value = proportions_ztest(count, nobs)

                results.append({
                    "dimension": dim,
                    "dimension_value": dim_value,
                    "test_number": test_id,
                    "metric": metric['name'],
                    "conversion_rate_1": conv1,
                    "conversion_rate_2": conv2,
                    "metric_change": conv2 / conv1 if conv1 > 0 else np.nan,
                    "z_stat": z_stat,
                    "p_value": p_value,
                    "significant": p_value < 0.05,
                    "n1": den1,
                    "n2": den2
                })

df_results = pd.DataFrame(results)

df_results.to_csv("test_results_dimension.csv", index=False)